In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import importlib
import transmission_functions as tf
importlib.reload(tf)
from glob import glob
import scipy as sp
from time import time
import seaborn as sns
from matplotlib import pyplot as plt
import networkx as nx
import polars as pl
import pickle
import glob
from joblib import Parallel, delayed
import oracledb

cmp = sns.color_palette('Set2')

use_edge_weights = True
settings = True
sensitivity = True
parallelise = False

n_cores = 10

print('Using edge weights: ' + str(use_edge_weights))
print('Prioritise all settings: ' + str(settings))
print('Running sensitivity analysis: ' + str(sensitivity))
print('Parallelised tree sampling: ' + str(parallelise) + ' on n_cores = ' + str(n_cores))

sl_idx = 0



file_path = '../Large_networks/*'
sl_list = ['Alpha_B.1.1++(Other_B.1.1)', 
           'Omicron_BA.1_like++(Probable_Omicron_BA.1_like_and_Unassigned)', 
           'Delta',
           'Eta_B.1.525_like++(Other)', 
           'wildtype']
sl_cluster_list = ['Alpha', 
                  'Omicron', 
                   'Delta',
                  'Eta', 
                  'wildtype']

sl = sl_list[sl_idx]
sl_cluster = sl_cluster_list[sl_idx]
print('Extracting trees for strain: ' + sl)
n_trees = 100

### Decisions about schools:
school_id_name = 'INSTNR' # Choose UDD_ID, INSTNR or INSTNR + UDD
school_types = pl.read_csv('../school_types.csv', has_header = False)
school_types = school_types.rename({'column_1' : 'school', 'column_2' : 'school_type'})
higher_ed = False # Include higher education or only schools and gymnasium


if sl == 'Delta':
    df_family_edgelist = pl.read_csv('../sliding_window_delta/sliding_window_data/df_family_edgelist.csv')
else:
    df_family_edgelist = pl.read_csv('../Large_Networks/' + sl + '/df_family_edgelist.csv')
family_pairs = df_family_edgelist.filter(~pl.col('PERSON_ID_1')
                                         .is_null()).select(pl.col('PERSON_ID_1', 'PERSON_ID_2')).to_numpy()

df_family_edgelist = df_family_edgelist.filter(~pl.col('PERSON_ID_1').is_null())


In [ ]:
family_array = df_family_edgelist.select(pl.col('PERSON_ID_1', 'PERSON_ID_2')).drop_nulls().to_numpy()
family_pairs_set = set([(e, v) for e, v in family_array] + [(v, e) for e, v in family_array])

### Generate Network 

In [ ]:

if sl == 'Delta':
    id_path = "../sliding_window_delta/graph_ids.csv"
else:
    id_path = "../Large_Networks/" + sl + "/" + sl + "_ids_full.csv" 
ids = pl.read_csv(id_path).select(pl.col('strain')).to_numpy().flatten()
ids_df = pl.read_csv(id_path).select(pl.col('strain'))
start = time()

if sl == 'Delta':
 
    infection_adj_mat = sp.sparse.load_npz("../sliding_window_delta/adjacency_delta.npz")
else:
    if sensitivity:
        infection_adj_mat = sp.sparse.load_npz("../Large_Networks/" + sl + "/" + sl +
                                           "weighted_adjacency_full_sparse_sensitivity.npz").T
    else:
        infection_adj_mat = sp.sparse.load_npz("../Large_Networks/" + sl + "/" 
                                               + 'weighted_adjacency_full_sparse.npz').T

end = time()
print('Sparse infection matrix read in: ' + str(np.round(end-start, 4)) + ' seconds!')
G_digraph = nx.DiGraph(infection_adj_mat, sequence_ids = ids)
end2 = time()
print('Digraph created in: ' + str(np.round(end2-end, 4)) + ' seconds!')


df_family_edgelist.filter(~(pl.col('PERSON_ID_1') == pl.col('PERSON_ID_2')))
family_pairs = df_family_edgelist.filter(~pl.col('PERSON_ID_1')
                                         .is_null()).select(pl.col('PERSON_ID_1', 'PERSON_ID_2')).to_numpy()

if sl == 'Delta':
    all_attributes = pl.read_csv('../sliding_window_delta/sliding_window_data/all_attributes.csv')
    all_attributes_work_school = pl.read_csv('../sliding_window_delta/sliding_window_data/all_attributes_work_school.csv')
else:
    all_attributes = pl.read_csv('../Large_Networks/' + sl + '/all_attributes.csv')
    all_attributes_work_school = pl.read_csv('../Large_Networks/' + sl + '/all_attributes_work_school.csv')

all_attributes_work_school = all_attributes_work_school.with_columns(pl.col('INSTNR', 'UDD').cast(str))
all_attributes_work_school = all_attributes_work_school.with_columns(school_id_new = pl.col('INSTNR'))

all_attributes_work_school = all_attributes_work_school.join(school_types,
                                                             left_on = 'UDD_ID',
                                                             right_on = 'school',
                                                             how = 'left')


address_ids = ids_df.join(all_attributes, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'UNIQUE_ADDRESS_ID')).to_numpy()

address_attr_dict = dict([(i, address_ids[i, 1]) for i in range(len(address_ids[:, 1]))])

PERSON_ids = ids_df.join(all_attributes, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'PERSON_ID')).to_numpy()

PERSON_ID_attr_dict = dict([(i, PERSON_ids[i, 1]) for i in range(len(PERSON_ids[:, 1]))])
nx.set_node_attributes(G_digraph, PERSON_ID_attr_dict, name = 'PERSON_ID')




if 'column_0' in all_attributes.columns:
    all_attributes = all_attributes.drop('column_0')
    
if 'column_0' in all_attributes_work_school.columns:
    all_attributes_work_school = all_attributes_work_school.drop('column_0')

school_ids = ids_df.join(all_attributes_work_school, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'school_id_new')).to_numpy()

school_attr_dict = dict([(i, school_ids[i, 1]) for i in range(len(school_ids[:, 1]))])

school_types = ids_df.join(all_attributes_work_school, 
                         on = 'strain', 
                          how = 'left').select(pl.col('strain', 
                                                      'school_type')).to_numpy()

school_types_attr_dict = dict([(i, school_types[i, 1]) for i in range(len(school_types[:, 1]))])

# Workplaces
work_ids = ids_df.join(all_attributes_work_school, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'ARB_NR')).to_numpy()

work_attr_dict = dict([(i, work_ids[i, 1]) for i in range(len(work_ids[:, 1]))])


kommune_ids = ids_df.join(all_attributes_work_school, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'KOMMUNE_NAME')).to_numpy()
kommune_attr_dict = dict([(i, kommune_ids[i, 1]) for i in range(len(kommune_ids[:, 1]))])

region_ids = ids_df.join(all_attributes_work_school, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'Region')).to_numpy()
region_attr_dict = dict([(i, region_ids[i, 1]) for i in range(len(region_ids[:, 1]))])

dates_ids = ids_df.join(all_attributes_work_school, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'SampleDateTime')).to_numpy()
dates_attr_dict = dict([(i, dates_ids[i, 1]) for i in range(len(dates_ids[:, 1]))])

nx.set_node_attributes(G_digraph, address_attr_dict, name = 'address')
nx.set_node_attributes(G_digraph, school_attr_dict, name = 'school')
nx.set_node_attributes(G_digraph, school_types_attr_dict, name = 'school_type')
nx.set_node_attributes(G_digraph, work_attr_dict, name = 'workplace')
nx.set_node_attributes(G_digraph, kommune_attr_dict, name = 'kommune')
nx.set_node_attributes(G_digraph, region_attr_dict, name = 'region')
nx.set_node_attributes(G_digraph, dates_attr_dict, name = 'SampleDateTime')
nodes_data = G_digraph.nodes(data = True)
for (start, end) in tqdm(G_digraph.edges):
    start_node = nodes_data[start]
    end_node = nodes_data[end]
    if (start_node['address'] == end_node['address']) and (end_node['address'] != None):
        nx.set_edge_attributes(G_digraph, {(start, end) : True}, name = 'share_address')
    else:
        nx.set_edge_attributes(G_digraph, {(start, end) : False}, name = 'share_address')
        
    if (start_node['kommune'] == end_node['kommune']) and (end_node['kommune'] != None):
        nx.set_edge_attributes(G_digraph, {(start, end) : True}, name = 'share_kommune')
    else:
        nx.set_edge_attributes(G_digraph, {(start, end) : False}, name = 'share_kommune')
        
    if (start_node['region'] == end_node['region']) and (end_node['region'] != None):
        nx.set_edge_attributes(G_digraph, {(start, end) : True}, name = 'share_region')
    else:
        nx.set_edge_attributes(G_digraph, {(start, end) : False}, name = 'share_region')
     
    
    if len(set([(start_node['PERSON_ID'],end_node['PERSON_ID']),
            (end_node['PERSON_ID'], start_node['PERSON_ID'])])
           .intersection(family_pairs_set)) > 0:
        
        nx.set_edge_attributes(G_digraph, {(start, end) : True}, name = 'family_relation')
    else:
        nx.set_edge_attributes(G_digraph, {(start, end) : False}, name = 'family_relation')


    if (start_node['school'] == end_node['school']) and (start_node['school_type'] == 'Grundskole' or 
         start_node['school_type'] == 'Gymnasiale uddannelser'):
        nx.set_edge_attributes(G_digraph, {(start, end) : True}, name = 'share_school')
    else:
        nx.set_edge_attributes(G_digraph, {(start, end) : False}, name = 'share_school')

    if start_node['workplace'] == end_node['workplace'] and end_node['workplace'] != None:
        nx.set_edge_attributes(G_digraph, {(start, end) : True}, name = 'share_workplace')
    else:
        nx.set_edge_attributes(G_digraph, {(start, end) : False}, name = 'share_workplace')


In [ ]:
# Count number of edges in each setting and compute proportion of overall graph
n_workplaces = len([(e, v) for e, v, d in tqdm(G_digraph.edges(data = True)) if (d['share_workplace'] == True)]) 

n_family = len([(e, v) for e, v, d in G_digraph.edges(data = True) if (d['family_relation'] == True) and
               (d['share_address'] == False)]) 
n_schools = len([(e, v) for e, v, d in G_digraph.edges(data = True) if (d['share_school'] == True)]) 
n_households = len([(e, v) for e, v, d in G_digraph.edges(data = True) if (d['share_address'] == True)]) 
 

In [ ]:
np.array((n_workplaces, n_family, n_schools, n_households)) / len(G_digraph.edges) * 100

In [ ]:
ids_attr_dict = dict([(i, PERSON_ids[i, 0]) for i in range(len(PERSON_ids[:, 0]))])

nx.set_node_attributes(G_digraph, 
                       ids_attr_dict,
                       name = 'strain')

sampledates_ids = ids_df.join(all_attributes_work_school, on = 'strain', 
                          how = 'left').select(pl.col('strain', 'SampleDateTime')).to_numpy()

sampledates_attr_dict = dict([(i, 
                               datetime.strptime(sampledates_ids[i, 1].split('T')[0], 
                               '%Y-%m-%d')) for i in range(len(school_ids[:, 1]))])

nx.set_node_attributes(G_digraph, 
                       sampledates_attr_dict,
                       name = 'SampleDateTime')

share_workplace_edges = [(u, v) for u, v, d 
                        in tqdm(G_digraph.edges(data = True)) 
                        if d['share_workplace'] == True]

share_school_edges = [(u, v) for u, v, d 
                     in tqdm(G_digraph.edges(data = True))
                     if d['share_school'] == True]
share_setting_edges = [(u, v) for u, v, d 
                     in tqdm(G_digraph.edges(data = True))
                     if (d['share_school'] == True) or 
                       (d['share_workplace'] == True) or
                      (d['share_address'] == True)]

share_all_setting_edges = [(u, v) for u, v, d 
                     in tqdm(G_digraph.edges(data = True))
                     if (d['share_school'] == True) or 
                       (d['share_workplace'] == True) or
                      (d['share_address'] == True) or
                        (d['family_relation'] == True)  ]

In [ ]:
workplace_graph = G_digraph.edge_subgraph(share_workplace_edges)
school_graph = G_digraph.edge_subgraph(share_school_edges)
setting_graph = G_digraph.edge_subgraph(share_all_setting_edges)
wcs_workplace = nx.weakly_connected_components(workplace_graph)
wcs_school = nx.weakly_connected_components(school_graph)
wcs_setting = nx.weakly_connected_components(setting_graph)

In [ ]:

wc_sizes_work = np.array([len(wc) for wc in tqdm(wcs_workplace)])
wc_sizes_school = np.array([len(wc) for wc in tqdm(wcs_school)])
wc_sizes_setting = np.array([len(wc) for wc in tqdm(wcs_setting)])

In [ ]:
# Get components belonging to a particular setting

def get_setting_component(component, setting):
    assert(setting in ['school', 'workplace', 'address', 'kommune', 'region', 'family'])
    setting_array = np.array([component.nodes[n][setting] for n in component])
    setting_array = setting_array[setting_array != None]
    unique, num = np.unique(setting_array, return_counts = True)
    return dict(zip(unique, num))

# Get number of individuals in a setting
def get_nsetting_component(component, setting):
    assert(setting in ['school', 'workplace', 'address', 'kommune', 'region'])
    setting_array = np.array([component.nodes[n][setting] for n in component])
    if setting in ['kommune', 'region']:
        setting_set = set(setting_array)
    else:
        setting_set = set(setting_array[~np.isnan(setting_array)])
    return len(setting_set)

# Get dates of tests within a cluster
def get_dates_component(component):
    dates_array = np.array([component.nodes[n]['SampleDateTime'] for n in component])
    ids_array = np.array([component.nodes[n]['strain'] for n in component])
    return dict(zip(ids_array, dates_array))

# Get start and end dates for a cluster
def get_first_last_dates_component(component, include_ids = True):
    dates_array = np.array([component.nodes[n]['SampleDateTime'] for n in component])
    dates_min_idx = np.argmin(dates_array)
    dates_max_idx = np.argmax(dates_array)
    first_date = dates_array[dates_min_idx]
    last_date = dates_array[dates_max_idx]
    ids_array = np.array([component.nodes[n]['strain'] for n in component])
    first_id = ids_array[dates_min_idx]
    last_id = ids_array[dates_max_idx]
    if include_ids:
        return {first_date : first_id, last_date : last_id}
    else: 
        return (first_date, last_date)
        
# Create a dictionary for each cluster with important attributes 
def compose_component_dict(component):
    component_dict = {}
    for setting in (['school', 'workplace', 'address', 'kommune', 'region']):
        component_dict[setting] = get_setting_component(component, setting)
        component_dict[setting] = get_setting_component(component, setting)
    component_dict['start_end_dates'] = get_first_last_dates_component(component, include_ids = True)
    return component_dict


In [ ]:
def get_settings_list(component_dict, setting):
    if setting in ['school', 'workplace']:
        return [val for key, val in largest_component_dict[setting].items() if ~np.isnan(key)]
    else:
        return [val for key, val in largest_component_dict[setting].items()]

# Get all pairs that cross into different regions and different municipalities ('kommuner' in Danish)
def get_regions_crossover_lists(component_graph):
    component_kommune_crossovers_ids = [(a,b) for a, b in component_graph.edges() 
                                         if (component_graph.nodes[a]['kommune'] != component_graph.nodes[b]['kommune'])
                                        and (component_graph.nodes[a]['kommune'] != None) 
                                        and (component_graph.nodes[b]['kommune'] != None)]
    component_kommune_crossovers_person_ids = [(component_graph.nodes[a]['PERSON_ID'], component_graph.nodes[b]['PERSON_ID'])
                                   for (a, b) in component_kommune_crossovers_ids]
    component_kommune_crossovers = [(component_graph.nodes[a]['kommune'], component_graph.nodes[b]['kommune'])
                                   for (a, b) in component_kommune_crossovers_ids]
    component_region_crossovers = [(component_graph.nodes[a]['region'], component_graph.nodes[b]['region'])
                                   for (a, b) in component_kommune_crossovers_ids]
    return component_kommune_crossovers_ids, component_kommune_crossovers, component_region_crossovers
    
    

### Generate dictionaries of information for each cluster and then save the resulting information

In [ ]:
save = 1
all_components_dict = {}
all_components_dict_graph = {}
i=0
size_cutoff = 1
crossover_dict =  {}
for component in tqdm(nx.weakly_connected_components(setting_graph)):
    if len(component) < size_cutoff:
        continue
    component_list = list(component)
    component_graph = setting_graph.subgraph(component_list)
    
    component_dict = compose_component_dict(component_graph)
    
    if i==0:
        clusters_df = pd.DataFrame.from_dict(dict(component_graph.nodes(data = True)), 
                               orient = 'index')
        clusters_df['cluster_number'] = i
    else:
        cluster_df_i = pd.DataFrame.from_dict(dict(component_graph.nodes(data = True)), 
                               orient = 'index')
        cluster_df_i['cluster_number'] = i
        clusters_df = pd.concat([clusters_df, cluster_df_i])
    pd.DataFrame.from_dict(dict(component_graph.nodes(data = True)), 
                           orient = 'index').to_csv('../clusters/' +
                                                    sl_cluster + 
                                                    '/node_attributes/nodelist_cluster_' + str(i) + '.csv')
    
    component_dict['component_size'] = len(component_graph)
   
    all_components_dict[i] = component_dict
    crossover_ids, crossover_kommuner, crossover_regioner = get_regions_crossover_lists(component_graph)
    if len(crossover_ids) > 0:
        crossover_dict[i] = {'PERSON_IDs' : crossover_ids, 
                            'crossover_kommuner' : crossover_kommuner, 
                            'crossover_regioner' : crossover_regioner}
    component_region_crossovers = [(component_graph.nodes[a]['region'], 
                                      component_graph.nodes[b]['region']) 
                                     for a, b in component_graph.edges() 
                                     if component_graph.nodes[a]['region'] != component_graph.nodes[b]['region']]
    component_kommune_crossovers = [(component_graph.nodes[a]['kommune'], 
                                      component_graph.nodes[b]['kommune']) 
                                     for a, b in component_graph.edges() 
                                     if component_graph.nodes[a]['kommune'] != component_graph.nodes[b]['kommune']]
    all_components_dict_graph[i] = component_dict
    i+=1
if save:
    clusters_df.to_csv('../clusters/' +
                        sl_cluster + 
                        '/node_attributes/nodelist_clusters_no_size_cutoff.csv')

    sp.sparse.save_npz('../clusters/' +
                        sl_cluster + 
                        '/adjacency_matrices/adjacency_settings_macro_school.npz', nx.to_scipy_sparse_array(setting_graph))


    np.savetxt('../clusters/' +
                        sl_cluster + 
                        '/adjacency_matrices/settings_network_ids_no_size_cutoff.csv', 
              np.array([d['strain'] for n, d in setting_graph.nodes(data = True)]).astype(str), 
              fmt = '%s')

    with open('../clusters/' +
                        sl_cluster + '/cluster_dicts/settings_cluster_dict_macro_school.pickle', 'wb') as handle:
        pickle.dump(all_components_dict, handle, protocol = pickle.HIGHEST_PROTOCOL)
    with open('../clusters/' +
                        sl_cluster + '/cluster_dicts/kommune_crossover_dict_macro_school.pickle', 'wb') as handle:
        pickle.dump(crossover_dict, handle, protocol = pickle.HIGHEST_PROTOCOL)


In [ ]:
import scienceplots
plt.style.use('science')
sl_cluster_list_new = ['wildtype', 'Alpha', 'Eta', 'Delta', 'Omicron']
cmp = plt.get_cmap('Dark2')
color_dict = {'School + Workplace' : cmp(0),
             'School Only' : cmp(1), 
             'Workplace Only' : cmp(2), 
             'Households only' : cmp(3), 
             'Small cluster' : cmp(4)}
plt.figure(figsize = (15, 5))
cumsum_clusters = 0
nclusters = 0
cumsum_clusters_list = []
cumsum_clusters_list_upper = []
fontsize = 12
n_school_work = 0
for sl_cluster in sl_cluster_list_new:
    if sl_cluster == 'Eta':
        continue
    
    cluster_types = []
    with open('../../clusters/' + sl_cluster + '/cluster_dicts/settings_cluster_dict.pickle', 'rb') as handle:
            all_components_dict = pickle.load(handle)
            
    cumsum_clusters += nclusters   
    cumsum_clusters_list += [cumsum_clusters]
    nclusters = len(all_components_dict.keys())
    cumsum_clusters_list_upper += [cumsum_clusters + nclusters]
    
    first_date_clusters = np.zeros((nclusters, 2)).astype(datetime)
    for i, (key, val) in enumerate(all_components_dict.items()):
        first_date_clusters[i, :] = np.array(list(val['start_end_dates'].keys()))
    first_date_clusters = first_date_clusters[first_date_clusters[:, 0].argsort()]    
    if sl_cluster == 'wildtype':
        first_cluster =np.min(first_date_clusters)
    if sl_cluster == 'Omicron':
        last_cluster =np.max(first_date_clusters)
        
    for i in tqdm(range(len(all_components_dict))):
        component_dict = all_components_dict[i]
        max_school = get_max_setting_values(component_dict, 'school')
        max_workplace = get_max_setting_values(component_dict, 'workplace')
        if max_school > 5 and max_workplace > 5:
            component_dict['type_cluster'] = 'School + Workplace'
        elif max_school > 5:
            component_dict['type_cluster'] = 'School Only'
        elif max_workplace > 5:
            component_dict['type_cluster'] = 'Workplace Only'
        elif max_school == 1 and max_workplace == 1:
            component_dict['type_cluster'] = 'Households only'
        else:
            component_dict['type_cluster'] = 'Small cluster'
        
    for i, (key, val) in enumerate(all_components_dict.items()):
        cluster_types += [val['type_cluster']] 
    ndays_alpha =  (np.max(first_date_clusters[:, 1]) - np.min(first_date_clusters[:, 0])).days
    plot_days_alpha = np.array([np.datetime64(np.min(first_date_clusters[:, 1])) +
                       np.timedelta64(i, 'D') for i in range(ndays_alpha)])

    for i in range(nclusters):
        if cluster_types[i] == 'Small cluster' or cluster_types[i] == 'Households only':
            continue
        n_school_work+=1
        plt.hlines(i + cumsum_clusters, np.datetime64(first_date_clusters[i, 0]), 
                   np.datetime64(first_date_clusters[i, 1]), 
                  linewidth = 1, color = color_dict[cluster_types[i]],
                   label = cluster_types[i].capitalize(), alpha = 0.4)
        
        
    plt.axhline(cumsum_clusters, color = 'black', linestyle = '--')
    handles, labels = plt.gca().get_legend_handles_labels()
    labels, ids = np.unique(labels, return_index = True)
    handles = [handles[i] for i in ids]
    plt.legend(handles, labels, bbox_to_anchor = [0.01, 0.1], title = 'Cluster Type')
    

    plt.grid(alpha = 0.5)
    plt.title('Duration and Type of Settings-based Clusters', fontsize = 14)
    plt.xlabel('Date', fontsize = 12)
    plt.ylabel('Cluster number', fontsize = 12)
plt.xlim([np.datetime64(first_cluster),
          np.datetime64(last_cluster) + np.timedelta64(40, 'D')])

# Annotations
i=0
for sl_cluster in (sl_cluster_list_new):
    if sl_cluster == 'Eta':
        continue
    if sl_cluster != 'Omicron':

        plt.annotate(sl_cluster.capitalize(), 
                     (np.datetime64(last_cluster) + np.timedelta64(int(40/5), 'D'), 
                      (cumsum_clusters_list[i] + cumsum_clusters_list_upper[i])/2), fontsize = fontsize)
    else:
        plt.annotate(sl_cluster.capitalize(), 
                     (np.datetime64(last_cluster) + np.timedelta64(int(40/5), 'D'), 
                      (cumsum_clusters_list[i] + 300)), fontsize = fontsize)
    i+=1
plt.gca().invert_yaxis()